# AutoGen 多智能体对话
## Week 2 Day 3 (05-13)：面向多智能体协作的对话抽象

**核心目标**：跑通 AutoGen 官方 quickstart，理解多智能体对话模式。

**和 LangGraph 的关键差异**：
- **LangGraph**（工程师视角）：显式状态机（Node + Edge），控制粒度细
- **AutoGen**（研究者视角）：智能体通过自然语言对话协作，行为更偏涌现


## 新手导读：AutoGen 更像“让多个角色开会”

AutoGen 的抽象和 LangGraph 很不一样。LangGraph 强调显式状态机；AutoGen 强调多个 Agent 通过自然语言消息协作。

核心词汇：

- `AssistantAgent`：一个带 system prompt、可调用模型的角色。
- `SelectorGroupChat`：由模型决定下一个该谁发言。
- Termination condition：什么时候停止对话，否则多 Agent 可能一直聊下去。
- Tool-enabled agent：某些角色可以带工具，不是所有角色都必须能执行动作。

阅读顺序建议：

1. 先看模型客户端工厂，所有 Agent 共享底层模型连接。
2. 再看双 Agent demo：理解“提问者”和“专家”是靠 prompt 区分角色。
3. 看 SelectorGroupChat：重点是 speaker selection，而不是固定流程。
4. 最后看工具 demo：多 Agent 里仍然可以接入工具，但协作方式变成对话式。

常见误区：

- 多 Agent 不等于一定更强；角色越多，成本和不可控性也越高。
- AutoGen 的行为更“涌现”，适合探索/讨论，不适合强审计流程。
- 如果你需要严格状态和审批，LangGraph 通常比 AutoGen 更合适。


## 1. 环境准备
导入 AutoGen 核心模块，并加载 API Key。

In [1]:
# AutoGen 的核心抽象：AssistantAgent（每个 Agent 只有一个 system_message 定义角色）。
# OpenAIChatCompletionClient 是 AutoGen 的 LLM 工厂，model_info 告诉框架该模型支持哪些能力。
import asyncio, json, math, os, sys
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

print("AutoGen 导入成功")


AutoGen 导入成功


## 2. 模型客户端工厂
AutoGen 0.7 使用 `OpenAIChatCompletionClient` 作为统一的 LLM 接口，也可以接入 OpenAI 兼容 API（包括 DeepSeek）。

In [2]:
# make_model_client() 每次创建新实例：AutoGen 里多个 Agent 共享模型端点，但各自维护对话历史。
# model_info 中 function_calling: True 是让 AutoGen 允许工具调用的关键——漏掉会静默失败。
def make_model_client():
    return OpenAIChatCompletionClient(
        model="deepseek-v4-flash",
        api_key=os.getenv("API_KEY"),
        base_url="https://api.deepseek.com/v1",
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": False,
            "family": "deepseek",
        },
    )

client = make_model_client()
print("模型客户端已创建")


模型客户端已创建


## 3. 核心概念：SelectorGroupChat

**AutoGen 的关键思想**：
1. **对话抽象**：智能体通过自然语言消息沟通，而不是显式状态迁移
2. **SelectorGroupChat**：由 LLM 充当“主持人”，动态选择下一位发言者
3. **涌现行为**：协作模式从对话中产生，而不是完全由预定义流程写死

**三种抽象哲学（面试 Q25）**：
- `LangGraph` -> 图抽象 -> “我明确知道流程是什么”
- `AutoGen` -> 对话抽象 -> “让智能体自己协商出结果”
- `CrewAI` -> 角色抽象 -> “每个人都有自己的职责”


## 4. Demo 1：双智能体对话（提问者 vs 专家）

**SelectorGroupChat** 会自动管理轮次，由 LLM 决定谁来发言。

这个最小示例展示了 AutoGen 的核心模式：
- 两个 `AssistantAgent` 实例彼此对话
- LLM selector 选择下一位发言者
- `TextMentionTermination` 在检测到特定文本时停止

In [3]:
# SelectorGroupChat 用 LLM 驱动发言者选择：每轮结束后 selector 看对话历史决定谁发言。
# selector_prompt 是关键：描述每个 Agent 的职责，帮助 selector LLM 做出正确决策。
# TextMentionTermination：某 Agent 输出包含指定关键词时终止，常用模式是约定 'DONE' / 'TERMINATE'。
# MaxMessageTermination：防止无限循环——生产环境必须设置兜底终止条件。
async def basic_demo():
    client = make_model_client()

    asker = AssistantAgent(
        name="asker",
        model_client=client,
        system_message="你是 'asker'，一名对 AI Agent 技术感兴趣的开发者。请用中文只问一个关于 LangGraph、AutoGen、CrewAI 框架选型的问题，不要输出其他内容。",
    )

    expert = AssistantAgent(
        name="expert",
        model_client=client,
        system_message="你是 'expert'，一名 AI 架构师。请用中文简洁回答（3-5 句），先给结论再解释，最后用 'DONE.' 结束。",
    )

    team = SelectorGroupChat(
        participants=[asker, expert],
        model_client=client,
        selector_prompt="先选择 asker 提问，再选择 expert 回答，最后选择 TERMINATE。",
        termination_condition=TextMentionTermination(text="DONE"),
    )

    result = await team.run(task="开始一段对话：asker 先提问，expert 再回答。")

    print("-" * 50)
    for msg in result.messages:
        if hasattr(msg, "source") and hasattr(msg, "content"):
            print(f"[{msg.source}]: {msg.content}")
            print()

    await client.close()

await basic_demo()


--------------------------------------------------
[user]: 开始一段对话：asker 先提问，expert 再回答。

[asker]: 我们被要求以asker身份只问一个问题，不要输出其他内容。问题关于LangGraph、AutoGen、CrewAI框架选型。所以直接输出问题。

[asker]: 在构建多智能体协作系统时，如何根据任务复杂度、智能体间通信模式以及可扩展性需求，在LangGraph、AutoGen和CrewAI之间做出合适的框架选型？

[expert]: 我们要求用中文简洁回答，先给结论再解释，3-5句，最后用'DONE.'结束。问题是关于多智能体协作系统框架选型。我将给出结论：根据任务复杂度、通信模式和可扩展性需求，推荐LangGraph用于高度结构化的工作流，AutoGen用于灵活对话与复杂交互，CrewAI用于快速原型和简单任务。然后解释每个框架的适用场景。

[expert]: 结论：根据任务复杂度、通信模式和可扩展性需求，推荐LangGraph用于高度结构化的工作流（如DAG依赖），AutoGen用于灵活对话与复杂交互（如代理间消息协商），CrewAI用于快速原型和简单任务（如固定角色分工）。具体而言，若任务具有明确执行顺序和状态机需求，LangGraph的图计算模型更优；若智能体间需要动态且密集的通信，AutoGen的对话循环机制更灵活；若仅需低耦合、即插即用的协作，CrewAI的声明式配置更简单。可扩展性上，LangGraph适合需要精确控制并发与状态的场景，AutoGen能处理不对称消息传递，而CrewAI在轻量级扩展下表现良好。DONE.



## 5. Demo 2：三智能体协作（研究员 + 作者 + 评论员）

这是 AutoGen 最有价值的使用场景：**通过自然语言实现多角色分工**。

三个智能体协作产出一篇短文：
1. **Researcher** -> 提供事实和论点
2. **Writer** -> 基于研究材料撰写文章
3. **Critic** -> 审阅并提出改进建议

这个模式对应 Project 3（multi-agent-collab）的核心思路。

In [4]:
# group_demo 演示多角色协作：researcher 收集事实，writer 撰写文章，critic 提出修改意见。
# SelectorGroupChat 让 LLM 决定轮次顺序，而不是写死 for 循环。
# 关键陷阱：如果没有明确终止词，对话可能在 researcher/writer/critic 之间无限循环。
async def group_demo():
    client = make_model_client()

    researcher = AssistantAgent(
        name="researcher", model_client=client,
        system_message="你是 'researcher'。拿到主题后，请用中文提供 3-5 条具体事实或论点。最后用 'RESEARCH_DONE' 结束。",
    )
    writer = AssistantAgent(
        name="writer", model_client=client,
        system_message="你是 'writer'。请基于研究材料，用中文写一篇 150-200 字短文（标题 + 正文 + 总结）。最后用 'WRITING_DONE' 结束。",
    )
    critic = AssistantAgent(
        name="critic", model_client=client,
        system_message="你是 'critic'。请用中文给出 2-3 条具体改进建议。最后用 'REVIEW_DONE_STOP' 结束。",
    )

    team = SelectorGroupChat(
        participants=[researcher, writer, critic],
        model_client=client,
        selector_prompt="顺序：researcher -> writer -> critic -> TERMINATE。不要重复选择同一个智能体。",
        termination_condition=TextMentionTermination(text="STOP"),
        max_turns=5,
    )

    result = await team.run(
        task="协作完成一篇短文：《为什么 AI Agent 需要工具调用》"
    )

    for msg in result.messages:
        if hasattr(msg, "source") and hasattr(msg, "content"):
            c = msg.content if isinstance(msg.content, str) else str(msg.content)[:300]
            print(f"[{msg.source}]: {c}")
            print()

    await client.close()

await group_demo()


[user]: 协作完成一篇短文：《为什么 AI Agent 需要工具调用》

[researcher]: 我们要求提供3-5条具体事实或论点，关于“为什么AI Agent需要工具调用”。需要以中文输出，最后用'RESEARCH_DONE'结束。作为researcher，提供简洁的事实或论点。

[researcher]: 1. **扩展能力边界**：AI Agent 的核心是自主决策与执行，但大语言模型本身无法直接操作外部系统（如数据库、API、文件系统）。工具调用使 Agent 能获取实时数据（如天气、股票价格）、执行计算（如数学运算）、访问专有知识库，从而突破模型静态知识限制，处理动态、结构化任务。  
2. **提高任务完成准确性与可靠性**：通过调用专用工具（如代码解释器、搜索引擎或表单验证器），Agent 可减少因模型幻觉产生的错误。例如，生成财务报表时，调用计算工具比依赖模型直接输出数字更少出错；执行多步骤流程时，工具调用能保证每一步结果可验证、可回滚。  
3. **实现复杂工作流自动编排**：单一调用仅解决单点问题，但 Agent 通过组合多个工具（如“先搜索用户信息→再调用邮件API发送通知”）可构建端到端自动化流程。工具调用充当“手脚”支撑了Agent的规划能力，使其能将抽象需求分解为具体操作链。  
4. **安全与权限控制**：工具调用允许开发者对 Agent 的操作施加细粒度规则（如调用数据库前需用户授权、限制写操作范围）。相比直接暴露底层系统，工具接口提供了审计日志、速率限制和权限校验，避免 Agent 越权或破坏性行为。  
5. **促进持续学习与适应**：通过工具调用的成功/失败反馈（如API返回错误代码、数据格式不匹配），Agent 可动态调整后续策略（如改用替代工具、修正调用参数）。这种反馈循环使 Agent 能适应环境变化，而无需重新训练模型。  

RESEARCH_DONE

[writer]: 我们基于研究材料写一篇短文，标题为《为什么 AI Agent 需要工具调用》。正文需要包含扩展能力边界、提高准确性、复杂工作流、安全控制、持续学习等要点。篇幅150-200字，最后用WRITING_DONE结束。

[writer]: ## 为什么 AI Agent 需要工具调用

大语言模型天然受限：知识静态、无法操作外部系统

## 6. 框架对比总结（面试 Q25 完整回答）

| 维度 | LangGraph | AutoGen | CrewAI |
|-----------|-----------|---------|--------|
| **核心抽象** | 图 | 对话 | 角色 |
| **设计哲学** | 工程化：显式 FSM | 研究型：涌现协作 | 业务型：角色建模 |
| **可控性** | 最高 | 中等 | 中等 |
| **学习曲线** | 最陡 | 中等 | 最容易 |
| **HITL 支持** | 原生支持 | 支持 | 相对有限 |
| **适合场景** | 生产系统、精细控制 | 多智能体研究、原型验证 | 业务演示、教学 |

### 选型指南（面试版）

1. **生产级核心 Agent** -> LangGraph（精细控制 + checkpoint + tracing）
2. **多智能体研究 / 原型验证** -> AutoGen（涌现对话 + 快速验证）
3. **业务演示 / 教学** -> CrewAI（角色直观 + 上手最快）
4. 遵循 Anthropic 原则：**先保持简单，只在确实需要时增加复杂度**

### 本周动手总结

| 日期 | 框架 | 关键收获 |
|-----|-----------|---------------|
| w2d1 | LangGraph | 把流程定义成图，精确但概念较重 |
| w2d2 | LangGraph HITL | `interrupt_before` 一行接入人工审批 |
| w2d3 | AutoGen | 让智能体决定谁发言，形成涌现式协作 |

**结论**：当你明确知道流程时，图抽象更合适；当你希望智能体自行协商时，对话抽象更合适。两者是互补关系。结合 Anthropic 的建议和我自己的经验，生产系统里我会优先用 LangGraph 做主编排；只有在确实需要多智能体协商时，才借鉴 AutoGen 的对话模式。但多数时候，一个设计良好、工具完善的单 Agent 已经足够。


## 学习检查清单

读完这节后，建议你能回答：

- AutoGen 和 LangGraph 的抽象差别是什么？
- SelectorGroupChat 是如何决定下一个发言者的？
- 为什么多 Agent 必须有终止条件？
- 什么场景适合 AutoGen 的对话式协作？
- 多 Agent 带来的主要成本和风险是什么？
